# LG 그램 쿠팡 리뷰 크롤링 & 부정(단점) 분석 노트북

이 노트북은 쿠팡 상품 페이지(예: LG 그램)의 사용자 리뷰를 수집한 뒤, 그 중 제품의 단점/불만 사항(발열, 소음, 배터리, 힌지 등)에 초점을 두어 전처리, 형태소 분석, 빈도/연관/시각화 분석을 수행합니다.

> 중요 고지 (Disclaimer)
> - 쿠팡 서비스 약관(ToS)과 robots.txt 정책을 반드시 사전 확인하세요.
> - 과도한 요청은 계정 차단/IP 블락을 야기할 수 있습니다. (폴라이트 크롤링: 느린 속도, 적은 양, 재시도 제한)
> - 본 노트북 코드는 교육/연구 목적 예시이며, 상업적/대량 수집 용도로 그대로 사용하지 마세요.
> - 가능하다면 공식 API / 제휴 경로를 우선 검토하세요.

아래 순서로 진행됩니다:
1) 환경 준비 및 라이브러리 설치
2) 설정(상수) 정의
3) 유틸 (지연, 백오프, 로깅)
4) Selenium 초기화
5) 리뷰 로딩 절차 구현
6) 리뷰 파싱 함수
7) 전체 리뷰 수집 루프
8) 중간 저장
9~20) 텍스트 전처리, 형태소, 감성/부정 키워드, n-gram, 다양한 시각화
21) 증분 크롤링 전략
22) 로깅/예외 처리 고도화
23) 간단한 단위 테스트
24) 속도 제어 & 정책 준수 안내

---


## 1. 라이브러리 설치 및 임포트

필요 라이브러리를 설치합니다. (Windows PowerShell 기준)
- selenium, webdriver-manager: 동적 로딩 페이지 자동화
- pandas, numpy: 데이터 처리
- beautifulsoup4: HTML 파싱 (보조)
- konlpy, nltk: 한국어 형태소/토큰 작업
- regex (내장 re로 충분하지만 일부 고급 패턴 예)
- wordcloud: 워드클라우드 생성
- networkx: 토큰 동시출현 네트워크
- tqdm: 진행바

(설치 셀은 재실행시 중복 설치를 피하려면 조건문을 둘 수 있음)


In [2]:
# (필요시) 패키지 설치 - 최초 1회
import importlib, sys, subprocess, pkgutil

required = [
    'selenium','webdriver-manager','pandas','beautifulsoup4','konlpy',
    'nltk','wordcloud','networkx','tqdm']

for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# 임포트
import os, re, json, time, math, random, logging, html, string, pickle, itertools
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options

# NLP
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# KoNLPy (Java 필요) -> Java 없을 때 우회용 더미 토크나이저 제공
_OKT_AVAILABLE = False
try:
    from konlpy.tag import Okt
    try:
        okt = Okt()
        _OKT_AVAILABLE = True
        print('Okt 형태소 분석기 로드 성공')
    except Exception as e:
        raise e
except Exception as e:
    print('[경고] KoNLPy Okt 사용 불가 (Java 미설치 또는 환경 문제). 단순 토크나이저로 대체합니다.')
    print('에러 상세:', e)
    # 간단한 fallback: 2글자 이상 한글 시퀀스를 명사로 가정
    class DummyOkt:
        def pos(self, text, stem=True):
            tokens = re.findall(r'[가-힣]{2,}', text)
            return [(t, 'Noun') for t in tokens]
    okt = DummyOkt()

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import networkx as nx

# Windows 한글폰트 (설치 여부 확인 후 설정)
FONT_CANDIDATES = [
    'C:/Windows/Fonts/malgun.ttf',
    'C:/Windows/Fonts/malgunbd.ttf'
]
for fp in FONT_CANDIDATES:
    if os.path.exists(fp):
        try:
            sns.set(font='Malgun Gothic', rc={'axes.unicode_minus':False})
        except Exception:
            pass
        break

print('환경 준비 완료 (Okt 사용 가능:' , _OKT_AVAILABLE, ')')

Installing webdriver-manager...
Installing beautifulsoup4...
Installing beautifulsoup4...
[경고] KoNLPy Okt 사용 불가 (Java 미설치 또는 환경 문제). 단순 토크나이저로 대체합니다.
에러 상세: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.
[경고] KoNLPy Okt 사용 불가 (Java 미설치 또는 환경 문제). 단순 토크나이저로 대체합니다.
에러 상세: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.
환경 준비 완료 (Okt 사용 가능: False )
환경 준비 완료 (Okt 사용 가능: False )


## 2. 설정: 대상 URL, 헤더, 경로 상수 정의

In [3]:
PRODUCT_URL = "https://www.coupang.com/vp/products/8733162257?itemId=23271242157&vendorItemId=93002332480"
OUTPUT_DIR = os.path.join(os.getcwd(), '03.CX_Group4','02.LG_Gram','outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:128.0) Gecko/20100101 Firefox/128.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 13_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.4 Safari/605.1.15'
]

MAX_REVIEWS = 500  # 안전: 필요시 조정
SCROLL_PAUSE = (1.5, 3.0)  # 범위 내 랜덤
CLICK_PAUSE = (1.0, 2.0)
RETRY_LIMIT = 5
BACKOFF_BASE = 1.6
RAW_JSONL = os.path.join(OUTPUT_DIR, 'gram_reviews_raw.jsonl')
PROCESSED_CSV = os.path.join(OUTPUT_DIR, 'gram_reviews_processed.csv')
NEGATIVE_CSV = os.path.join(OUTPUT_DIR, 'gram_reviews_negative.csv')
FREQ_CSV = os.path.join(OUTPUT_DIR, 'negative_token_frequency.csv')
NGRAM_CSV = os.path.join(OUTPUT_DIR, 'negative_ngrams.csv')
LOG_FILE = os.path.join(OUTPUT_DIR, 'crawler.log')

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('coupang_gram')
logger.info('설정 완료')

[2025-08-18 14:33:04,729] INFO 설정 완료


## 3. 지연·재시도 유틸 함수 작성

In [4]:
def rand_sleep(rng: tuple):
    dur = random.uniform(*rng)
    time.sleep(dur)


def with_retry(fn, *args, **kwargs):
    for attempt in range(1, RETRY_LIMIT+1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            wait = BACKOFF_BASE ** attempt + random.random()
            logger.warning(f"[with_retry] attempt={attempt} error={e} waiting {wait:.2f}s")
            time.sleep(wait)
    raise RuntimeError(f"Function {fn.__name__} failed after {RETRY_LIMIT} attempts")


def safe_find(driver, by, value, timeout=5):
    try:
        return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((by, value)))
    except TimeoutException:
        return None


def safe_click(driver, element):
    try:
        driver.execute_script("arguments[0].scrollIntoView(true);", element)
        rand_sleep(CLICK_PAUSE)
        element.click()
        return True
    except Exception as e:
        logger.debug(f'safe_click fail: {e}')
        return False


def scroll_to_bottom(driver, step_pause=1.5, max_iters=30):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_iters):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(step_pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

logger.info('유틸 함수 정의 완료')

[2025-08-18 14:33:06,882] INFO 유틸 함수 정의 완료


## 4. Selenium WebDriver 초기화 (Headless)

In [5]:
def init_driver(user_agent=None):
    ua = user_agent or random.choice(USER_AGENTS)
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1280,1800')
    options.add_argument(f'--user-agent={ua}')
    # options.add_argument('--blink-settings=imagesEnabled=false')  # 이미지 비활성 (선택)
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(40)
    return driver

# 드라이버 생성 (필요시 나중에 실행)
try:
    driver = init_driver()
    driver.get(PRODUCT_URL)
    logger.info('상품 페이지 접속')
except Exception as e:
    logger.error(f'Driver init 실패: {e}')

print('드라이버 준비')

[2025-08-18 14:33:08,921] INFO ====== WebDriver manager ======
[2025-08-18 14:33:09,839] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:09,839] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:09,886] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:09,886] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:09,922] INFO There is no [win64] chromedriver "139.0.7258.68" for browser google-chrome "139.0.7258" in cache
[2025-08-18 14:33:09,922] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:09,922] INFO There is no [win64] chromedriver "139.0.7258.68" for browser google-chrome "139.0.7258" in cache
[2025-08-18 14:33:09,922] INFO Get LATEST chromedriver version for google-chrome
[2025-08-18 14:33:10,018] INFO WebDriver version 139.0.7258.68 selected
[2025-08-18 14:33:10,021] INFO Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/139.0.7258.68

## 5. 전체 리뷰 로딩: 스크롤 & 더보기 버튼 반복

In [ ]:
def open_review_tab(driver):
    # 쿠팡 UI 변동 가능, 선택자 변경 필요할 수 있음
    candidates = [
        (By.CSS_SELECTOR, 'a.sdp-review__article__page__all'),
        (By.XPATH, "//a[contains(@href,'reviews')]"),
    ]
    for by_, sel in candidates:
        el = safe_find(driver, by_, sel, timeout=4)
        if el:
            safe_click(driver, el)
            rand_sleep((2,4))
            return True
    return False


def load_all_reviews(driver, max_reviews=MAX_REVIEWS):
    opened = open_review_tab(driver)
    if not opened:
        logger.warning('리뷰 탭 자동 전환 실패 (수동 확인 필요)')
    collected = 0
    same_count_iters = 0
    last_count = 0
    while collected < max_reviews:
        scroll_to_bottom(driver, step_pause=random.uniform(*SCROLL_PAUSE), max_iters=2)
        # 더보기 버튼 시도
        more_btn = safe_find(driver, By.CSS_SELECTOR, 'button.sdp-review__article__page__more')
        if more_btn:
            safe_click(driver, more_btn)
        time.sleep(random.uniform(*SCROLL_PAUSE))
        review_els = driver.find_elements(By.CSS_SELECTOR, 'section.sdp-review__article__list__review')
        collected = len(review_els)
        if collected == last_count:
            same_count_iters += 1
        else:
            same_count_iters = 0
        last_count = collected
        logger.info(f'현재 수집된 로딩된 리뷰 DOM 수: {collected}')
        if same_count_iters >= 3:
            logger.info('더 이상 증가 없음 -> 중단')
            break
    return driver.find_elements(By.CSS_SELECTOR, 'section.sdp-review__article__list__review')

# (선택 실행)
# review_elements = load_all_reviews(driver)
# len(review_elements)

## 6. 단일 리뷰 파싱 함수 구현

In [6]:
def parse_review(el):
    try:
        html_block = el.get_attribute('outerHTML')
        soup = BeautifulSoup(html_block, 'html.parser')
        # 별점
        star_el = soup.select_one('.sdp-review__article__list__info__product-info__star-orange')
        rating = None
        if star_el and star_el.get('data-rating'):  # 가정
            rating = float(star_el['data-rating'])
        else:
            # 대안: style width% 기반 추정
            style_el = soup.select_one('.sdp-review__article__list__info__product-info__star__active')
            if style_el and 'style' in style_el.attrs:
                # width:80% -> 4.0
                m = re.search(r'width\s*:\s*(\d+)%', style_el['style'])
                if m:
                    rating = round(float(m.group(1)) / 20, 1)
        # 날짜
        date_text = None
        date_el = soup.select_one('.sdp-review__article__list__info__product-info__reg-date')
        if date_el:
            date_text = date_el.get_text(strip=True)
        # 내용
        content_el = soup.select_one('.sdp-review__article__list__review__content')
        if not content_el:
            content_el = soup.select_one('.sdp-review__article__list__review__content__description')
        content = content_el.get_text(' ', strip=True) if content_el else ''
        # 구매인증
        verified = bool(soup.select_one('.sdp-review__article__list__info__badge'))
        # 옵션
        option_el = soup.select_one('.sdp-review__article__list__info__product-info__option')
        option_text = option_el.get_text(' ', strip=True) if option_el else ''
        review_id = hash(content + (date_text or ''))
        return {
            'review_id': review_id,
            'rating': rating,
            'date': date_text,
            'option': option_text,
            'content': content,
            'verified': verified,
            'raw_html': html_block
        }
    except Exception as e:
        logger.error(f'parse_review error: {e}')
        return None

logger.info('parse_review 정의 완료')

[2025-08-18 14:33:17,340] INFO parse_review 정의 완료


## 7. 전체 리뷰 수집 루프 (에러 처리 포함)

In [7]:
def collect_reviews(driver, max_reviews=MAX_REVIEWS, interim_save_every=50):
    review_elements = load_all_reviews(driver, max_reviews=max_reviews)
    collected = []
    seen = set()
    failures = 0
    for idx, el in enumerate(review_elements):
        if idx >= max_reviews:
            break
        data = parse_review(el)
        if not data:
            failures += 1
            continue
        if data['review_id'] in seen:
            continue
        seen.add(data['review_id'])
        collected.append(data)
        if (idx+1) % interim_save_every == 0:
            with open(RAW_JSONL, 'a', encoding='utf-8') as f:
                for r in collected[-interim_save_every:]:
                    f.write(json.dumps(r, ensure_ascii=False) + '\n')
            logger.info(f'중간 저장 {idx+1}건')
    # 최종 저장 append
    with open(RAW_JSONL, 'a', encoding='utf-8') as f:
        for r in collected:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    logger.info(f'총 수집: {len(collected)} (실패 {failures})')
    return collected

# (선택 실행)
# raw_reviews = collect_reviews(driver, max_reviews=MAX_REVIEWS)
# len(raw_reviews)

## 8. 리뷰 원시 데이터 저장 (중간 백업)
위 collect_reviews 함수 내부에서 일정 건마다 JSONL append 저장을 수행합니다. 필요시 여기서 재로딩/병합 로직을 작성할 수 있습니다.

## 9. 텍스트 정제: HTML 제거, 공백/이모지 처리

In [8]:
import html as ihtml
EMOJI_PATTERN = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)
TAG_PATTERN = re.compile('<[^>]+>')
MULTI_SPACE = re.compile('\s+')

DOMAIN_STOPWORDS = set(['쿠팡','배송','상품','사용','후기','그램','노트북','구매','제품','LG','그램북','사용자'])
PUNCTS = set(string.punctuation)

NEGATIVE_KEYWORDS = {
    '발열': ['발열','뜨겁','열나','온도','과열'],
    '팬소음': ['소음','팬소음','시끄럽','웽','팬 돌아','팬소리'],
    '배터리': ['배터리','충전','전원','배터리시간','배터리시간','배터리표시'],
    '성능': ['성능','버벅','느리','렉','프리징','느림'],
    '가격': ['가격','비싸','가성비','비용'],
    '힌지': ['힌지','접히','덜컹','유격'],
    '밝기': ['밝기','어둡','밝음','디스플레이밝기'],
    '키감': ['키감','키보드','눌림','키 travel','타건'],
    '내구성': ['내구','약하','깨지','헐거','튼튼하지','스크래치'],
    '스피커': ['스피커','음질','소리 작','사운드 약'],
    '화면': ['화면','패널','디스플레이','빛샘','각도','시야각'],
}
NEG_FLAT = set(itertools.chain.from_iterable(NEGATIVE_KEYWORDS.values()))

SENT_SPLIT = re.compile(r'[.!?\n]+')


def clean_text(text: str) -> str:
    t = ihtml.unescape(text)
    t = TAG_PATTERN.sub(' ', t)
    t = EMOJI_PATTERN.sub(' ', t)
    t = re.sub('[^\w\sㄱ-힣.,!?-]', ' ', t)
    t = MULTI_SPACE.sub(' ', t).strip()
    return t


def remove_stopwords(tokens):
    out = []
    for tok in tokens:
        if tok in DOMAIN_STOPWORDS: continue
        if tok in PUNCTS: continue
        if tok.isdigit(): continue
        if len(tok) == 1: continue
        out.append(tok)
    return out


def contains_negative(sentence: str):
    for base, kws in NEGATIVE_KEYWORDS.items():
        for kw in kws:
            if kw in sentence:
                return True
    return False


def negative_categories(sentence: str):
    cats = []
    for base, kws in NEGATIVE_KEYWORDS.items():
        if any(kw in sentence for kw in kws):
            cats.append(base)
    return cats


def tokenize(text: str):
    # 명사 + 형용사
    pos = okt.pos(text, stem=True)
    tokens = [w for w, p in pos if p in ['Noun','Adjective']]
    tokens = [t.lower() for t in tokens]
    tokens = remove_stopwords(tokens)
    return tokens


def ngrams(tokens, n=2):
    return ['_'.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

logger.info('전처리/분석 함수 정의 완료')

[2025-08-18 14:33:21,543] INFO 전처리/분석 함수 정의 완료


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:31: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:31: SyntaxWarning: invalid escape sequence '\w'
C:\Users\lgdx\AppData\Local\Temp\ipykernel_18676\2217697972.py:4: SyntaxWarning: invalid escape sequence '\s'
  MULTI_SPACE = re.compile('\s+')
C:\Users\lgdx\AppData\Local\Temp\ipykernel_18676\2217697972.py:31: SyntaxWarning: invalid escape sequence '\w'
  t = re.sub('[^\w\sㄱ-힣.,!?-]', ' ', t)


## 10~14. (위 코드 셀에서 정의됨)

## 15. 단점 관련 n-gram (bi/tri) 추출 & 16. 감성 점수 부여 & 17~19. 시각화 & 20. 최종 저장

In [9]:
# (실행 전) raw_reviews가 메모리에 없다면 JSONL 로드
if 'raw_reviews' not in globals():
    raw_reviews = []
    if os.path.exists(RAW_JSONL):
        with open(RAW_JSONL, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    raw_reviews.append(json.loads(line))
                except: pass
    logger.info(f'로드된 raw 리뷰 수: {len(raw_reviews)}')

# DataFrame
raw_df = pd.DataFrame(raw_reviews)
if raw_df.empty:
    logger.warning('raw_df 비어있음 - 먼저 크롤 수행 필요')
else:
    raw_df['clean_content'] = raw_df['content'].astype(str).apply(clean_text)
    # 문장 분할 -> 부정 문장 여부, 카테고리
    sentences = []
    for rid, txt in zip(raw_df['review_id'], raw_df['clean_content']):
        for sent in SENT_SPLIT.split(txt):
            s = sent.strip()
            if not s: continue
            neg = contains_negative(s)
            cats = negative_categories(s) if neg else []
            sentences.append({
                'review_id': rid,
                'sentence': s,
                'is_negative_sentence': neg,
                'negative_categories': cats
            })
    sent_df = pd.DataFrame(sentences)
    # 리뷰 단위 negative flag
    neg_review_ids = set(sent_df[sent_df['is_negative_sentence']]['review_id'])
    raw_df['has_negative'] = raw_df['review_id'].isin(neg_review_ids)

    # 토큰화 (전체 / 부정 문장만)
    sent_df['tokens'] = sent_df['sentence'].apply(tokenize)
    neg_sent_df = sent_df[sent_df['is_negative_sentence']].copy()

    # 단점 관련 토큰만 필터
    def filter_negative_tokens(tokens):
        return [t for t in tokens if any(t.startswith(k[:2]) or k in t for k in NEG_FLAT)]
    neg_sent_df['neg_tokens'] = neg_sent_df['tokens'].apply(filter_negative_tokens)

    # 빈도수 집계
    all_neg_tokens = list(itertools.chain.from_iterable(neg_sent_df['neg_tokens']))
    freq = Counter(all_neg_tokens)
    freq_df = pd.DataFrame(freq.most_common(), columns=['token','count'])

    # n-grams
    bigrams = Counter()
    trigrams = Counter()
    for toks in neg_sent_df['neg_tokens']:
        bigrams.update(ngrams(toks,2))
        trigrams.update(ngrams(toks,3))
    bigram_df = pd.DataFrame([(k,v) for k,v in bigrams.items() if v>=2], columns=['bigram','count']).sort_values('count', ascending=False)
    trigram_df = pd.DataFrame([(k,v) for k,v in trigrams.items() if v>=2], columns=['trigram','count']).sort_values('count', ascending=False)

    # 감성 점수 (단순: 부정 키워드 등장 수 * -1)
    def sentiment_score(text):
        score = 0
        for kw in NEG_FLAT:
            if kw in text:
                score -= 1
        return score
    raw_df['sentiment_score'] = raw_df['clean_content'].apply(sentiment_score)

    # 시각화 1: 상위 토큰 막대
    topN = 20
    plt.figure(figsize=(8,6))
    sns.barplot(data=freq_df.head(topN), x='count', y='token', palette='Reds_r')
    plt.title('상위 부정 토큰 빈도')
    plt.tight_layout()
    plt.show()

    # 시각화 2: 워드클라우드
    if all_neg_tokens:
        wc = WordCloud(width=800, height=400, background_color='white', font_path='C:/Windows/Fonts/malgun.ttf')
        wc.generate_from_frequencies(freq)
        plt.figure(figsize=(12,5))
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title('부정 토큰 워드클라우드')
        plt.show()

    # 시각화 3: 동시출현 네트워크
    co_counts = Counter()
    for toks in neg_sent_df['neg_tokens']:
        uniq = list(set(toks))
        for a, b in itertools.combinations(sorted(uniq), 2):
            co_counts[(a,b)] += 1
    # 필터 최소 공출현 2
    edges = [(a,b,w) for (a,b), w in co_counts.items() if w>=2]
    if edges:
        G = nx.Graph()
        for a,b,w in edges:
            G.add_edge(a,b,weight=w)
        plt.figure(figsize=(10,8))
        pos = nx.spring_layout(G, k=0.8, seed=42)
        weights = [G[u][v]['weight'] for u,v in G.edges]
        nx.draw_networkx_nodes(G,pos,node_size=600,node_color='salmon')
        nx.draw_networkx_edges(G,pos,width=[w for w in weights],alpha=0.4)
        nx.draw_networkx_labels(G,pos,font_family='Malgun Gothic')
        plt.title('부정 토큰 동시출현 네트워크 (>=2)')
        plt.axis('off')
        plt.show()

    # 저장
    raw_df.to_csv(PROCESSED_CSV, index=False, encoding='utf-8-sig')
    neg_sent_df.to_csv(NEGATIVE_CSV, index=False, encoding='utf-8-sig')
    freq_df.to_csv(FREQ_CSV, index=False, encoding='utf-8-sig')
    bigram_df.to_csv(NGRAM_CSV.replace('.csv','_bigram.csv'), index=False, encoding='utf-8-sig')
    trigram_df.to_csv(NGRAM_CSV.replace('.csv','_trigram.csv'), index=False, encoding='utf-8-sig')
    logger.info('저장 완료')

[2025-08-18 14:33:23,853] INFO 로드된 raw 리뷰 수: 0
[2025-08-18 14:33:23,870] WARNING raw_df 비어있음 - 먼저 크롤 수행 필요
[2025-08-18 14:33:23,870] WARNING raw_df 비어있음 - 먼저 크롤 수행 필요


## 21. 증분 크롤링: 마지막 페이지/오프셋 체크
## 22. 로깅 & 예외 처리 개선
## 23. 단위 테스트: 파서/정제 함수
## 24. 속도 제어 & 서비스 약관 준수 확인 주석

In [10]:
# 21. 증분 크롤링: 기존 review_id 집합 로드 후 신규만 append

def load_existing_ids(path=RAW_JSONL):
    ids = set()
    if os.path.exists(path):
        with open(path,'r',encoding='utf-8') as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    ids.add(rec.get('review_id'))
                except: pass
    return ids


def incremental_collect(driver, max_new=100):
    existing = load_existing_ids()
    logger.info(f'기존 review id 수: {len(existing)}')
    review_elements = load_all_reviews(driver, max_reviews=max_new+len(existing))
    new_count = 0
    with open(RAW_JSONL,'a',encoding='utf-8') as f:
        for el in review_elements:
            data = parse_review(el)
            if not data: continue
            if data['review_id'] in existing: continue
            f.write(json.dumps(data, ensure_ascii=False)+'\n')
            existing.add(data['review_id'])
            new_count += 1
            if new_count >= max_new:
                break
    logger.info(f'증분 신규 수집: {new_count}')

# 22. 로깅은 상단 logging.basicConfig로 처리. 필요시 logger.setLevel(logging.DEBUG)

# 23. 단위 테스트 (간단): pytest 없이 간단 assert 기반

def _test_clean_text():
    sample = '<div>안녕😀 테스트!!!</div>'
    cleaned = clean_text(sample)
    assert 'div' not in cleaned
    assert '안녕' in cleaned
    print('clean_text OK')

def _test_negative_detection():
    s = '발열이 있고 팬소음이 매우 심합니다'
    assert contains_negative(s) is True
    cats = negative_categories(s)
    assert '발열' in cats and '팬소음' in cats
    print('negative detection OK')

def _test_tokenize():
    toks = tokenize('배터리가 빨리 닳고 발열 때문에 불편')
    assert isinstance(toks, list) and len(toks)>0
    print('tokenize OK')

if __name__ == '__main__':
    _test_clean_text()
    _test_negative_detection()
    _test_tokenize()

# 24. 속도 제어 & 정책 준수 메모:
# - MAX_REVIEWS, max_new 등을 낮게 유지 (예: 300~500 이하)
# - 각 클릭/스크롤 사이 rand_sleep 적용
# - 야간시간 과도수집 금지, 병렬 스레드/프로세스 사용 지양
# - 최신 쿠팡 robots.txt, 약관 재확인 필요

print('섹션 21~24 실행 준비 완료')

clean_text OK
negative detection OK
tokenize OK
섹션 21~24 실행 준비 완료


## 추가 감정(부정 강도) 시각화
다음 셀은 기존 `raw_df`와 `sent_df` (필요시 재생성)를 활용해 감정 분포와 카테고리별 평균 점수 등을 시각화합니다.

In [12]:
# 감정(부정) 시각화: 분포, 카테고리 평균, 시간 추이
import matplotlib.dates as mdates

# 데이터 확보
if 'raw_df' not in globals() or raw_df.empty:
    if os.path.exists(PROCESSED_CSV):
        raw_df = pd.read_csv(PROCESSED_CSV)
    else:
        print('데이터가 없어 감정 시각화를 건너뜁니다. 먼저 크롤 & 처리 셀 실행 후 다시 시도하세요.')
        raw_df = pd.DataFrame()

if raw_df.empty:
    # 조기 종료
    print('[INFO] raw_df 비어있음 - 시각화 스킵')
else:
    # sentiment_score 없으면 재계산
    if 'sentiment_score' not in raw_df.columns:
        if 'clean_content' not in raw_df.columns:
            raw_df['clean_content'] = raw_df['content'].astype(str).apply(clean_text)
        def sentiment_score(text):
            score = 0
            for kw in NEG_FLAT:
                if kw in str(text):
                    score -= 1
            return score
        raw_df['sentiment_score'] = raw_df['clean_content'].apply(sentiment_score)

    # 문장 레벨 데이터 필요 시 재구성
    if 'sent_df' not in globals():
        sentences = []
        for rid, txt in zip(raw_df['review_id'], raw_df['clean_content']):
            for sent in SENT_SPLIT.split(str(txt)):
                s = sent.strip()
                if not s: continue
                neg = contains_negative(s)
                cats = negative_categories(s) if neg else []
                sentences.append({'review_id': rid,'sentence': s,'is_negative_sentence': neg,'negative_categories': cats})
        sent_df = pd.DataFrame(sentences)

    # 1) 리뷰 단위 sentiment_score 히스토그램
    plt.figure(figsize=(7,4))
    if raw_df['sentiment_score'].nunique() > 1:
        bins = range(int(raw_df['sentiment_score'].min())-1, int(raw_df['sentiment_score'].max())+2)
    else:
        bins = 5
    plt.hist(raw_df['sentiment_score'], bins=bins, color='tomato', edgecolor='black')
    plt.title('리뷰 감정 점수 분포 (<=0: 부정 수치가 낮을수록 부정 강함)')
    plt.xlabel('sentiment_score (음수=부정)')
    plt.ylabel('리뷰 수')
    plt.tight_layout()
    plt.show()

    # 2) 부정 카테고리별 등장 리뷰 수 / 평균 점수
    cat_counts = []
    neg_sent_only = sent_df[sent_df['is_negative_sentence']]
    if not neg_sent_only.empty:
        for rid, group in neg_sent_only.groupby('review_id'):
            cats = set(itertools.chain.from_iterable(group['negative_categories']))
            for c in cats:
                cat_counts.append({'review_id': rid, 'category': c})
    cat_df = pd.DataFrame(cat_counts)
    if not cat_df.empty:
        merged = cat_df.merge(raw_df[['review_id','sentiment_score']], on='review_id', how='left')
        agg = merged.groupby('category').agg(review_cnt=('review_id','nunique'), avg_sent=('sentiment_score','mean')).reset_index()
        fig, axes = plt.subplots(1,2, figsize=(12,4))
        sns.barplot(data=agg.sort_values('review_cnt', ascending=False), x='review_cnt', y='category', ax=axes[0], palette='Reds_r')
        axes[0].set_title('카테고리별 부정 리뷰 수')
        sns.barplot(data=agg.sort_values('avg_sent'), x='avg_sent', y='category', ax=axes[1], palette='coolwarm')
        axes[1].set_title('카테고리별 평균 감정 점수 (더 낮음=부정 강함)')
        plt.tight_layout()
        plt.show()
    else:
        print('부정 카테고리 데이터 없음')

    # 3) 시간 추이 (날짜 컬럼이 파싱 가능할 때)
    if 'date' in raw_df.columns and raw_df['date'].notna().any():
        def parse_date(x):
            for fmt in ('%Y.%m.%d','%Y-%m-%d','%Y/%m/%d','%y.%m.%d'):
                try:
                    return datetime.strptime(str(x).strip(), fmt)
                except Exception:
                    continue
            return pd.NaT
        if 'parsed_date' not in raw_df.columns:
            raw_df['parsed_date'] = raw_df['date'].apply(parse_date)
        ts = raw_df.dropna(subset=['parsed_date']).groupby(pd.Grouper(key='parsed_date', freq='D')).agg(mean_sent=('sentiment_score','mean'), cnt=('review_id','count')).reset_index()
        if not ts.empty:
            fig, ax1 = plt.subplots(figsize=(10,4))
            ax2 = ax1.twinx()
            ax1.plot(ts['parsed_date'], ts['mean_sent'], color='red', marker='o', label='평균 감정')
            ax2.bar(ts['parsed_date'], ts['cnt'], alpha=0.3, color='gray', label='리뷰 수')
            ax1.set_ylabel('평균 감정(음수=부정)')
            ax2.set_ylabel('리뷰 수')
            ax1.set_title('일자별 평균 감정 & 리뷰 수')
            ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
            fig.autofmt_xdate()
            lines, labels = [], []
            for ax in (ax1, ax2):
                line_lbl = ax.get_legend_handles_labels()
                lines += line_lbl[0]; labels += line_lbl[1]
            ax1.legend(lines, labels, loc='best')
            plt.tight_layout()
            plt.show()
        else:
            print('유효 날짜 데이터 부족')
    else:
        print('날짜 컬럼 없음 또는 값 없음 -> 시간 추이 스킵')

    print('추가 감정 시각화 완료')

데이터가 없어 감정 시각화를 건너뜁니다. 먼저 크롤 & 처리 셀 실행 후 다시 시도하세요.
[INFO] raw_df 비어있음 - 시각화 스킵


## LG 공식 사이트 리뷰 크롤링 (별점 낮은 순)
LG전자 공식 제품 페이지 리뷰를 별점 낮은 순으로 수집합니다. (페이지 구조/선택자는 변경될 수 있으므로 개발자 도구(F12)로 실제 클래스명을 확인 후 필요시 수정하세요.)

주의:
- 서비스 약관 및 로봇 배제정책을 준수.
- 과도한 요청/빠른 스크롤 지양.
- 별점 낮은 순 정렬 UI 요소가 로딩된 뒤 정렬을 '낮은 평점'으로 전환.


In [13]:
# LG 공식 사이트 리뷰 크롤링 (별점 낮은 순)
# NOTE: 실제 DOM 구조는 수시 변경 가능. 아래 selectors는 예시이며 필요시 개발자도구로 재확인하십시오.
# 절차:
#  1) 페이지 이동
#  2) 리뷰 영역 탭 활성화 (필요 시)
#  3) 정렬 드롭다운 열기 -> '별점 낮은순' 클릭
#  4) 페이지네이션 반복 / '더보기' 클릭 반복
#  5) 각 리뷰에서 별점/날짜/제목/내용 추출

LG_PRODUCT_URL = "https://www.lge.co.kr/care-solutions/notebook/16z90tp-ka5wk?dpType=careTab"
LG_LOW_RATING_JSONL = os.path.join(OUTPUT_DIR, 'lg_official_low_rating_reviews.jsonl')

# 추정 선택자 (실제 확인 필요)
SEL_SORT_TRIGGER = 'button.sort__btn'            # 정렬 버튼
SEL_SORT_OPTIONS = 'ul.sort__list li'            # 옵션 리스트 li
SORT_KEYWORDS = ['별점','낮','별점 낮']          # 옵션 텍스트 내 포함 단어 후보
SEL_REVIEW_CONTAINER = 'div.review__list-item'   # 단일 리뷰 블록
SEL_REVIEW_RATING = '.review__score'             # 별점 숫자/아이콘 영역
SEL_REVIEW_DATE = '.review__date'
SEL_REVIEW_TITLE = '.review__title'
SEL_REVIEW_CONTENT = '.review__text'
SEL_MORE_BUTTON = 'button.review__more'          # 더보기 (페이지 추가)


def init_lg_driver():
    return init_driver()


def open_low_rating_sort(driver):
    # 정렬 버튼 클릭 -> 옵션 중 '별점 낮은순' 찾기
    trig = safe_find(driver, By.CSS_SELECTOR, SEL_SORT_TRIGGER, timeout=6)
    if trig:
        safe_click(driver, trig)
        time.sleep(1)
        opts = driver.find_elements(By.CSS_SELECTOR, SEL_SORT_OPTIONS)
        for o in opts:
            txt = o.text.strip()
            if all(k in txt for k in ['별점','낮']) or any(k in txt for k in SORT_KEYWORDS):
                safe_click(driver, o)
                time.sleep(2)
                return True
    return False


def parse_lg_review(el):
    try:
        html_block = el.get_attribute('outerHTML')
        soup = BeautifulSoup(html_block, 'html.parser')
        rating = None
        rating_el = soup.select_one(SEL_REVIEW_RATING)
        if rating_el:
            # 숫자 추출 (예: ★★☆☆☆ or '2/5')
            txt = rating_el.get_text(' ', strip=True)
            m = re.search(r'(\d+(?:\.\d+)?)', txt)
            if m:
                rating = float(m.group(1))
        date_el = soup.select_one(SEL_REVIEW_DATE)
        date_txt = date_el.get_text(strip=True) if date_el else ''
        title_el = soup.select_one(SEL_REVIEW_TITLE)
        title = title_el.get_text(' ', strip=True) if title_el else ''
        content_el = soup.select_one(SEL_REVIEW_CONTENT)
        content = content_el.get_text('\n', strip=True) if content_el else ''
        rid = hash(title + content + date_txt)
        return {
            'review_id': rid,
            'rating': rating,
            'date': date_txt,
            'title': title,
            'content': content,
            'source': 'LG_OFFICIAL'
        }
    except Exception as e:
        logger.debug(f'parse_lg_review error: {e}')
        return None


def crawl_lg_low_ratings(max_pages=5, max_reviews=300):
    drv = init_lg_driver()
    drv.get(LG_PRODUCT_URL)
    time.sleep(3)
    opened = open_low_rating_sort(drv)
    if not opened:
        print('[경고] 낮은 별점 정렬 선택 실패 - 기본 정렬로 진행됩니다.')
    results = []
    seen = set()
    page = 0
    while page < max_pages and len(results) < max_reviews:
        time.sleep(2)
        review_els = drv.find_elements(By.CSS_SELECTOR, SEL_REVIEW_CONTAINER)
        for el in review_els:
            data = parse_lg_review(el)
            if not data: continue
            if data['review_id'] in seen: continue
            seen.add(data['review_id'])
            results.append(data)
            if len(results) >= max_reviews:
                break
        # 더보기 시도
        more_btn = safe_find(drv, By.CSS_SELECTOR, SEL_MORE_BUTTON, timeout=3)
        if more_btn and safe_click(drv, more_btn):
            page += 1
            continue
        else:
            break
    # 저장
    with open(LG_LOW_RATING_JSONL, 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False)+'\n')
    print(f'LG 공식 낮은 평점 리뷰 수집 완료: {len(results)}건 저장 -> {LG_LOW_RATING_JSONL}')
    drv.quit()
    return results

# 예시 실행 (원할 때 주석 해제)
# lg_low_reviews = crawl_lg_low_ratings(max_pages=3, max_reviews=150)
# len(lg_low_reviews)